### In the AIDev dataset, Human PRs only exist for repos ≥500 stars. The AIDev-pop only has agent data for repos ≥500 stars. We started with the Human data, and then combined the same repositories that existed from all_repository

In [ ]:
!pip install polars pyarrow --quiet
import polars as pl

# URLs
human_pr_url = "https://huggingface.co/datasets/hao-li/AIDev/resolve/main/human_pull_request.parquet"
all_pr_url    = "https://huggingface.co/datasets/hao-li/AIDev/resolve/main/all_pull_request.parquet"
repo_url      = "https://huggingface.co/datasets/hao-li/AIDev/resolve/main/all_repository.parquet"

# ----------------------------
# 1. Load data
# ----------------------------
print("Loading datasets...")
human_pr = pl.read_parquet(human_pr_url)
all_pr   = pl.read_parquet(all_pr_url)
repos    = pl.read_parquet(repo_url)

print("Human PR columns:", human_pr.columns)
print("All PR columns:", all_pr.columns)

# ----------------------------
# 1B. Fill missing repo_id for human PRs
# repos.url   = repo_url
# repos.id    = repo_id
# ----------------------------
human_pr = human_pr.join(
    repos.select(["url", "id"]).rename({"url": "repo_url", "id": "repo_id"}),
    on="repo_url",
    how="left"
)

print("After filling, human repo_id null count:",
      human_pr["repo_id"].null_count())

# ---------------------------------------------------------
# Drop human PRs whose repos are missing from repos table
# ---------------------------------------------------------
missing_urls = (
    human_pr
    .filter(pl.col("repo_id").is_null())
    .select("repo_url")
    .unique()
)

missing_list = missing_urls["repo_url"].to_list()

print("\n===== These repos exist in human PRs but NOT in all_repository / all_PR datasets =====")
for url in missing_list:
    print(url)

print(f"\nDropping {len(missing_list)} repos from human_pr...")

# Drop rows where repo_id is null
human_pr = human_pr.filter(pl.col("repo_id").is_not_null())

print("New human_pr shape after dropping:", human_pr.shape)

note = {
    "dropped_repo_urls": missing_list,
    "reason": "These repos appear in human PR dataset but not in all_repository or agent PR datasets."
}

print("\nNOTE:", note)

# ----------------------------
# 2. Identify repos used in Human PR dataset
# ----------------------------
human_repo_ids = human_pr.select("repo_url").unique()
print("\nUnique human repo URLs:", human_repo_ids.height)

if "repo_id" in human_pr.columns:
    human_repo_ids = human_pr.select("repo_id").unique()

# ----------------------------
# 3. Filter agent PRs to those repos
# ----------------------------
print("\nFiltering agent PRs to human PR repos...")

agent_pr = all_pr.filter(pl.col("agent") != "Human")

agent_pr_matched = agent_pr.join(
    human_pr.select("repo_url").unique(),
    on="repo_url",
    how="inner"
)

print("Filtered agent PRs:", agent_pr_matched.shape)

# ----------------------------
# 4. Align schemas before concat
# ----------------------------
print("\n===== Columns BEFORE alignment =====")
print("Human PR columns:", human_pr.columns)
print("Agent PR (matched) columns:", agent_pr_matched.columns)

human_cols = set(human_pr.columns)
agent_cols = set(agent_pr_matched.columns)

missing_in_human = agent_cols - human_cols
missing_in_agent = human_cols - agent_cols

print("Missing in human:", missing_in_human)
print("Missing in agent:", missing_in_agent)

# Add missing cols to human_pr
for col in missing_in_human:
    human_pr = human_pr.with_columns(pl.lit(None).alias(col))

# Add missing cols to agent_pr
for col in missing_in_agent:
    agent_pr_matched = agent_pr_matched.with_columns(pl.lit(None).alias(col))

# Reorder
agent_pr_matched = agent_pr_matched.select(sorted(agent_pr_matched.columns))
human_pr         = human_pr.select(sorted(human_pr.columns))

# ----------------------------
# 5. Concat
# ----------------------------
final_dataset = pl.concat([human_pr, agent_pr_matched], how="vertical_relaxed")

print("\n===== Columns AFTER merge =====")
print("Final dataset columns:", final_dataset.columns)
print("Final dataset size:", final_dataset.shape)

# ----------------------------
# 6. Save to local file
# ----------------------------
output_path = "human_agent_same_repos.parquet"
final_dataset.write_parquet(output_path)

print(f"\nSaved unified dataset to: {output_path}")

Uploaded this to https://huggingface.co/datasets/kaylamarietorres/AIDev/human_agent_same_repos.parquet